Πως ελέγχεις τις "δυνάμεις" του υπολογιστή σου για να δεις τι περιθώρια έχεις για PyTorch projects. Αυτός ο κώδικας είναι το κλασικό πρώτο βήμα για κάθε data scientist: η επιβεβαίωση ότι το hardware είναι σωστά ρυθμισμένο.

Τι κάνει ο κώδικάς σου:
Διαθεσιμότητα CUDA: Ελέγχει αν οι οδηγοί (drivers) της NVIDIA είναι εγκατεστημένοι και προσβάσιμοι.

Μοντέλο GPU: Σου λέει ακριβώς ποια κάρτα γραφικών έχεις (π.χ. NVIDIA RTX 3060).

Χωρητικότητα VRAM: Μετατρέπει τα bytes της συνολικής μνήμης σε Gigabytes (GB).

📝 Επεξήγηση: Έλεγχος Κατάστασης Μνήμης GPU (VRAM)
Αυτός ο κώδικας χρησιμοποιείται για να παρακολουθούμε πώς η PyTorch διαχειρίζεται τους πόρους της κάρτας γραφικών σε πραγματικό χρόνο.

torch.cuda.is_available(): Επιβεβαιώνει ότι οι οδηγοί (drivers) και το CUDA toolkit είναι σωστά ρυθμισμένα.

memory_allocated (Πραγματική Χρήση):

Αντιπροσωπεύει τη μνήμη που καταλαμβάνουν οι Tensors και τα Μοντέλα που έχουμε φορτώσει αυτή τη στιγμή.

Αν διαγράψουμε ένα αντικείμενο (del), αυτή η τιμή μειώνεται.

memory_reserved (Προκρατημένη Μνήμη):

Είναι η συνολική μνήμη που έχει "δανειστεί" η PyTorch από το λειτουργικό σύστημα (Caching Allocator).

Ακόμα κι αν ελευθερώσουμε έναν tensor, η PyTorch κρατάει αυτή τη μνήμη δεσμευμένη για να μπορεί να τη 
χρησιμοποιήσει ακαριαία στην επόμενη πράξη, χωρίς να καθυστερεί ζητώντας ξανά άδεια από τον driver της NVIDIA.

In [5]:
import torch

if torch.cuda.is_available():
    # Τρέχουσα δεσμευμένη μνήμη από την PyTorch
    allocated = torch.cuda.memory_allocated(0) / 1e9
    # Μνήμη που έχει δεσμευτεί στην cache του driver
    reserved = torch.cuda.memory_reserved(0) / 1e9
    
    print(f"Allocated Memory: {allocated:.2f} GB")
    print(f"Cached/Reserved: {reserved:.2f} GB")
    
#> **Σημείωση:** Αν το `Allocated` είναι 0.00, σημαίνει ότι η GPU είναι έτοιμη αλλά δεν έχουμε στείλει ακόμα δεδομένα σε αυτήν.

Allocated Memory: 0.00 GB
Cached/Reserved: 0.00 GB


### 🔍 Advanced GPU Diagnostic
Αυτή η συνάρτηση εκτελεί έναν πλήρη έλεγχο στο hardware:
* **Versions**: Επιβεβαιώνει τη συμβατότητα Python/Torch/CUDA.
* **Architecture**: Εμφανίζει το CUDA Capability (σημαντικό για optimization).
* **VRAM Status**: Υπολογίζει την πραγματική ελεύθερη μνήμη που απομένει για το μοντέλο σας.

In [6]:
import torch
import sys

def gpu_report():
    print(f"--- Σύστημα & Εκδόσεις ---")
    print(f"Python Version: {sys.version.split()[0]}")
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA Available: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"\n--- Στοιχεία GPU ---")
        device_id = torch.cuda.current_device()
        props = torch.cuda.get_device_properties(device_id)
        
        print(f"ID Συσκευής: {device_id}")
        print(f"Μοντέλο: {props.name}")
        print(f"CUDA Capability: {props.major}.{props.minor}")
        print(f"Συνολική VRAM: {props.total_memory / 1e9:.2f} GB")
        print(f"Multi-Processor Count: {props.multi_processor_count}")
        
        # Έλεγχος τρέχουσας χρήσης
        used = torch.cuda.memory_allocated(device_id) / 1e9
        reserved = torch.cuda.memory_reserved(device_id) / 1e9
        free = (props.total_memory / 1e9) - used
        
        print(f"\n--- Κατάσταση Μνήμης ---")
        print(f"Χρησιμοποιείται (Allocated): {used:.2f} GB")
        print(f"Δεσμευμένη (Reserved): {reserved:.2f} GB")
        print(f"Ελεύθερη (Περίπου): {free:.2f} GB")
    else:
        print("\n⚠️ Προσοχή: Η PyTorch δεν μπορεί να βρει την GPU σου.")

gpu_report()

--- Σύστημα & Εκδόσεις ---
Python Version: 3.12.7
PyTorch Version: 2.5.1+cu121
CUDA Available: True

--- Στοιχεία GPU ---
ID Συσκευής: 0
Μοντέλο: NVIDIA GeForce GTX 1650 Ti
CUDA Capability: 7.5
Συνολική VRAM: 4.29 GB
Multi-Processor Count: 16

--- Κατάσταση Μνήμης ---
Χρησιμοποιείται (Allocated): 0.00 GB
Δεσμευμένη (Reserved): 0.00 GB
Ελεύθερη (Περίπου): 4.29 GB
